In [ ]:
# Notebooks live in notebooks/; make the repo root importable (ppo, train_ppo, ...).
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ppo" / "paths.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


# `plot_trajectories` — usage walkthrough

Overlay FBS rollout trajectories on common axes.

Inputs the plotter accepts (auto-detected):

| spec | resolves to |
|------|-------------|
| `test_logs/<X-Y-Z>/<stem>` | auto-appends `_trajectory.csv` |
| `test_logs/<X-Y-Z>/<stem>_trajectory.csv` | rollout (MultiIndex CSV) |
| `<run_dir>/evals/<ts>/` | first `*_trajectory.csv` found in the directory |
| any flat CSV | needs `fbs<i>_x` / `fbs<i>_y` columns |

Each spec is a bare path, a `(path, label)` pair, or a `(path, label, group)`
triple. Trajectories sharing a **group** share a colormap and a linestyle, which
is how you contrast two policies at a glance.

**Constraint:** the plotter refuses to mix 1-FBS and 2-FBS trajectories in one
figure (raises `ValueError`).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from plot_trajectories import (
    plot_trajectories,
    load_trajectory,
    REPO_ROOT,
    TEST_LOGS,
)

## 1. What is available to plot

In [ ]:
print("Rollout trajectories under test_logs/ (by code):")
for code_dir in sorted(p for p in TEST_LOGS.glob("*") if p.is_dir()):
    trajs = sorted(code_dir.glob("*_trajectory.csv"))
    print(f"  [{code_dir.name}]")
    for t in trajs[-3:]:
        print("    ", t.name.replace("_trajectory.csv", ""))

print("\nEval trajectories under ppo_runs/*/evals/:")
from ppo.paths import RUNS_DIR
for run in sorted(RUNS_DIR.glob("*"))[-5:]:
    for t in sorted(run.glob("evals/*/ep*_trajectory.csv"))[:2]:
        print("  ", t.relative_to(RUNS_DIR))

## 2. Inspect a single trajectory

`load_trajectory` normalizes any supported input into the same shape.

In [ ]:
# Point this at one of the paths printed above.
SPEC = "test_logs/1-1-1/run_044_20260501_124857"

traj = load_trajectory(SPEC)
print("name      :", traj.name)
print("group     :", traj.group)
print("num_fbs   :", traj.num_fbs)
print("code      :", traj.code)
print("mbs (x,y) :", traj.mbs_x, traj.mbs_y)
print("shape     :", traj.df.shape)
traj.df.head()

## 3. One trajectory

In [ ]:
plot_trajectories([(SPEC, "policy")], title="Single rollout")
plt.show()

## 4. Compare two policies

Give each spec its own **group** (third tuple element). Groups get separate
colormaps and linestyles, so the two sets stay readable even when the paths
overlap. Both specs must share `num_fbs` or the plotter raises.

In [ ]:
specs = [
    ("test_logs/1-1-1/run_044_20260501_124857", "baseline", "a"),
    ("test_logs/1-1-1/run_039_20260501_124158", "shaped",   "b"),
]

fig, ax = plot_trajectories(
    specs,
    title="baseline vs shaped  (1-1-1)",
    color_by="power",
    mark_power_status=True,
)
plt.show()

## 5. Seed variance within one policy

Leave everything in the same group: the shared colormap shades each run
differently while keeping them visually a single family.

In [ ]:
specs = [
    ("test_logs/1-1-1/run_044_20260501_124857", "seed 0"),
    ("test_logs/1-1-1/run_039_20260501_124158", "seed 1"),
]
plot_trajectories(specs, title="Repeat rollouts")
plt.show()

## 6. Color encodings

`color_by='identity'` (default), `'power'` (colorbar in W), or `'step'`
(colorbar over the step index).

In [ ]:
fig, ax = plot_trajectories(
    [(SPEC, "policy")],
    title="Colored by step",
    color_by="step",
    figsize=(7, 5.5),
    annotate_endpoints=True,
)
plt.show()

## 7. Pinning colormaps per group

In [ ]:
fig, ax = plot_trajectories(
    [
        ("test_logs/1-1-1/run_044_20260501_124857", "A", "a"),
        ("test_logs/1-1-1/run_039_20260501_124158", "B", "b"),
    ],
    group_cmaps={"a": "Greens", "b": "Purples"},
    title="Explicit group colormaps",
)
plt.show()

## 8. Side-by-side panels

Pass an existing `ax` to place a plot into your own figure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

plot_trajectories(
    [("test_logs/1-1-1/run_044_20260501_124857", "baseline")],
    title="baseline",
    ax=axes[0],
)
plot_trajectories(
    [("test_logs/1-1-1/run_039_20260501_124158", "shaped")],
    title="shaped",
    ax=axes[1],
)

fig.tight_layout()
out = REPO_ROOT / "trajectory_panels.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print("saved ->", out)
plt.show()

## 9. The num_fbs guard

In [ ]:
# Replace these with two runs that genuinely have different num_fbs to see the
# guard fire. Otherwise just leave it commented out.

# try:
#     plot_trajectories([
#         ("test_logs/1-1-1/run_044_20260501_124857", "1 FBS"),
#         ("test_logs/2-1-1/run_045_20260501_125458", "2 FBS"),
#     ])
# except ValueError as e:
#     print("Guard fired:", e)

## 10. CLI equivalent

Same plot, no notebook:

```bash
python plot_trajectories.py \
    test_logs/1-1-1/run_044_20260501_124857_trajectory.csv \
    test_logs/1-1-1/run_039_20260501_124158_trajectory.csv \
    --label baseline --label shaped \
    --group a --group b \
    --title "baseline vs shaped  (1-1-1)" \
    --save trajectory_compare.png
```